# Inference Demo: Fine-tuned SQL Generator

Load the fine-tuned model from HuggingFace Hub and try it on new questions.

In [1]:
import sys
sys.path.append('..')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_REPO = "mattiinn/sql-generation-tinyllama"

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
)

print("Loading LoRA adapter from Hub...")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()

print("Model ready for inference")

c:\Users\Snapp\anaconda3\envs\sql-finetune\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 201.44it/s]


Loading LoRA adapter from Hub...
Model ready for inference


## Try your own questions

Edit the schema and question below to test the model.

In [2]:
from src.data.prompt_formatter import SQLPromptFormatter
from src.data.dataset_loader import SQLExample

formatter = SQLPromptFormatter()

def generate_sql(question: str, schema: str) -> str:
    example = SQLExample(question=question, sql="", db_id="custom", schema=schema)
    prompt  = formatter.format_inference(example)

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return formatter.extract_sql(generated)

In [3]:
schema = """Table: employees (id: number, name: text, department: text, salary: number)
Table: departments (id: number, name: text, budget: number)"""

questions = [
    "How many employees are there?",
    "What is the average salary?",
    "List all employees in the Engineering department",
]

for q in questions:
    sql = generate_sql(q, schema)
    print(f"Q: {q}")
    print(f"SQL: {sql}")
    print()

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How many employees are there?
SQL: SELECT count(*) FROM employees



[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the average salary?
SQL: SELECT avg(salary) FROM departments

Q: List all employees in the Engineering department
SQL: SELECT eid FROM employees JOIN departments AS T1 ON eid = T2.department_eid JOIN departments AS T2 ON T1.department_id = T2.id WHERE T1.name = "Engineering"



## Limitations Observed

This model (fine-tuned on 1000 examples, 3 epochs) is a proof-of-concept
demonstrating the QLoRA/LoRA fine-tuning pipeline end-to-end, not a
production-ready text-to-SQL system. See notebook 03 for full evaluation
results and honest limitations including table-name hallucination and
occasional query logic errors.

To productionize this, the next steps would be training on the full
7000-example Spider set with early stopping, and validating with a
proper schema-aware execution accuracy check.